In [2]:
%pip install -U gradio requests python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import time
import requests
import gradio as gr

from dotenv import load_dotenv


# Load variables from the .env file
load_dotenv()


# Read the OpenRouter API key
OPENROUTER_API_KEY = (
    os.getenv("OPENROUTER_API_KEY")
    or os.getenv("OPENROUTER_KEY")
)

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

# Confirmed working model
MODEL = "google/gemini-3.6-flash"


# Only the required headers
headers = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json"
}


SYSTEM_PROMPT = (
    "You are JEKACODE AI Code Explainer. "
    "Explain code accurately and clearly. "
    "Use simple language for beginners. "
    "Explain what the code does, its main logic, inputs, and outputs. "
    "Mention bugs or improvements when relevant. "
    "Keep the explanation concise and useful. "
    "Use Markdown headings and code blocks when helpful. "
    "Do not invent what the code does. "
    "Do not rewrite the entire code unless requested."
)


def explain_code(
    code,
    programming_language,
    explanation_type,
    experience_level,
    extra_request
):
    start_time = time.perf_counter()

    # Check API key
    if not OPENROUTER_API_KEY:
        return (
            "### Configuration error\n\n"
            "OpenRouter API key was not found. "
            "Please check your `.env` file."
        )

    # Check code input
    if not code or not str(code).strip():
        return "### Please enter some code first."

    code_text = str(code).strip()

    if extra_request:
        extra_text = str(extra_request).strip()
    else:
        extra_text = "None"

    user_prompt = (
        "Programming language: "
        + str(programming_language)
        + "\n\n"
        "Developer experience: "
        + str(experience_level)
        + "\n\n"
        "Requested explanation type: "
        + str(explanation_type)
        + "\n\n"
        "Additional request: "
        + extra_text
        + "\n\n"
        "Code to explain:\n"
        "```"
        + str(programming_language).lower()
        + "\n"
        + code_text
        + "\n```\n\n"
        "Explain the code clearly and accurately."
    )

    payload = {
        "model": MODEL,
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        "temperature": 0.2,
        "max_tokens": 350
    }

    try:
        response = requests.post(
            OPENROUTER_URL,
            headers=headers,
            json=payload,
            timeout=8
        )

        response.raise_for_status()

        data = response.json()

        answer = data["choices"][0]["message"]["content"]

        elapsed_time = time.perf_counter() - start_time

        return (
            str(answer)
            + "\n\n---\n\n"
            + "**Response time:** `"
            + f"{elapsed_time:.2f}"
            + " seconds`"
        )

    except requests.exceptions.Timeout:
        return (
            "### Request timed out\n\n"
            "The AI provider took too long to respond. "
            "Please try again."
        )

    except requests.exceptions.HTTPError:
        try:
            error_details = response.json()
        except Exception:
            error_details = response.text

        return (
            "### API error\n\n"
            "```text\n"
            + str(error_details)
            + "\n```"
        )

    except KeyError:
        return (
            "### Unexpected API response\n\n"
            "The response did not contain the expected answer."
        )

    except Exception as error:
        return (
            "### Unexpected error\n\n"
            "`"
            + str(error)
            + "`"
        )


print("OpenRouter key loaded:", bool(OPENROUTER_API_KEY))
print("Model:", MODEL)
print("Function ready:", callable(explain_code))

OpenRouter key loaded: True
Model: google/gemini-3.6-flash
Function ready: True


In [5]:
test_code = """
def greet(name):
    return f"Hello, {name}"
"""

test_result = explain_code(
    code=test_code,
    programming_language="Python",
    explanation_type="Simple explanation",
    experience_level="Beginner",
    extra_request=""
)

print(test_result)

Here is a simple explanation of the code:

### What This Code

---

**Response time:** `4.93 seconds`


In [6]:
EXAMPLE_CODE = """def calculate_total(price, quantity):
    subtotal = price * quantity
    tax = subtotal * 0.075
    total = subtotal + tax
    return total


amount = calculate_total(2000, 3)
print(amount)
"""


with gr.Blocks(
    title="JEKACODE AI Code Explainer"
) as demo:

    gr.Markdown(
        """
        # JEKACODE AI Code Explainer

        Understand code faster with clear explanations,
        logic breakdowns, bug detection, and improvement suggestions.
        """
    )

    with gr.Row():

        with gr.Column(scale=1):

            programming_language = gr.Dropdown(
                choices=[
                    "Python",
                    "JavaScript",
                    "TypeScript",
                    "HTML",
                    "CSS",
                    "SQL",
                    "Java",
                    "C",
                    "C++",
                    "C#",
                    "PHP",
                    "Go",
                    "Rust",
                    "Dart",
                    "Kotlin",
                    "Other"
                ],
                value="Python",
                label="Programming language"
            )

            explanation_type = gr.Dropdown(
                choices=[
                    "Simple explanation",
                    "Line-by-line explanation",
                    "Explain the main logic",
                    "Find bugs and errors",
                    "Suggest improvements",
                    "Explain like I am a beginner",
                    "Prepare me for an interview"
                ],
                value="Simple explanation",
                label="Explanation type"
            )

            experience_level = gr.Radio(
                choices=[
                    "Beginner",
                    "Intermediate",
                    "Advanced"
                ],
                value="Beginner",
                label="Your experience level"
            )

            extra_request = gr.Textbox(
                label="Additional request",
                placeholder=(
                    "Example: Explain the loop carefully."
                ),
                lines=3
            )

        with gr.Column(scale=2):

            code_input = gr.Code(
                label="Paste your code here",
                language="python",
                value=EXAMPLE_CODE,
                lines=18
            )

            with gr.Row():

                explain_button = gr.Button(
                    "Explain Code",
                    variant="primary"
                )

                example_button = gr.Button(
                    "Load Example"
                )

                clear_button = gr.Button(
                    "Clear"
                )

    result_output = gr.Markdown(
        label="AI Explanation"
    )

    explain_button.click(
        fn=explain_code,
        inputs=[
            code_input,
            programming_language,
            explanation_type,
            experience_level,
            extra_request
        ],
        outputs=result_output
    )

    example_button.click(
        fn=lambda: EXAMPLE_CODE,
        inputs=None,
        outputs=code_input
    )

    clear_button.click(
        fn=lambda: ("", ""),
        inputs=None,
        outputs=[
            code_input,
            result_output
        ]
    )

In [7]:
demo.launch(
    inbrowser=True,
    share=False
)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
